# Lithography Ops AI — Stage A: Semantic RAG (Google Colab)

This notebook adds **true semantic retrieval (RAG)** to the LithoOps AI project.

It runs entirely on **Colab's free machine** (so it doesn't need your laptop's RAM),
uses only **free, open tools**, and needs **no API keys**:

- **sentence-transformers** (`all-MiniLM-L6-v2`) — a small local embedding model
- **FAISS** — a fast, free vector store

**What "semantic" means:** instead of matching exact words, we match *meaning*.
A query like *"machine getting hot and blurry"* will find the cooling / focus
documents even though those exact words never appear in them.

> All maintenance documents here are **synthetic** — invented for this educational
> prototype from general public knowledge. Nothing is real ASML data.

**How to use:** click each cell and press ▶ (or Shift+Enter), top to bottom.

## 1. Install the free tools

Takes ~1 minute on Colab.

In [ ]:
!pip -q install sentence-transformers faiss-cpu
print('Installed.')

## 2. Load the synthetic knowledge base

These 29 invented maintenance documents are embedded directly in the notebook, so there is nothing to upload.

In [ ]:
KNOWLEDGE = [
  {
    "doc_id": "KB-OVL-01",
    "subsystem": "reticle stage",
    "title": "Overlay error drift diagnosis",
    "content": "Rising overlay error over several hours commonly indicates reticle stage calibration drift rather than a sudden fault. Begin by reviewing the stage position sensor trend and comparing left/right mark residuals. Re-run the overlay calibration sequence; if residuals persist above 3 nm, inspect the reticle stage sensor (part P-STG-04) for contamination or aging. Confirm the correction model has not saturated. Escalate to a stage specialist if drift resumes within one shift of recalibration."
  },
  {
    "doc_id": "KB-OVL-02",
    "subsystem": "reticle stage",
    "title": "Reticle stage vibration signature analysis",
    "content": "Elevated vibration on the reticle stage typically appears first as increased high-frequency content during acceleration phases. Capture a vibration spectrum and look for peaks near the stage resonance band. Persistent broadband vibration suggests bearing wear or a loose counterbalance; narrow peaks suggest a control-loop tuning issue. Cross-check with overlay residuals, since stage vibration frequently degrades overlay before it trips any alarm."
  },
  {
    "doc_id": "KB-OVL-03",
    "subsystem": "reticle stage",
    "title": "Overlay calibration procedure",
    "content": "Standard overlay recalibration: place the calibration reticle, run the mark detection routine across all field points, and let the system compute the correction grid. Verify the reported model error is within specification before releasing the tool. If the routine fails mark detection repeatedly, the illumination on the alignment sensor may be low; clean the sensor window and retry before replacing hardware."
  },
  {
    "doc_id": "KB-COOL-01",
    "subsystem": "cooling",
    "title": "Cooling system temperature rise",
    "content": "Temperature and focus error rising together is a classic signature of a cooling fault. The thermal expansion from inadequate cooling shifts the focal plane, so focus error tracks the temperature climb. Inspect the pump seals (part P-COOL-01), verify coolant flow rate against the nominal setpoint, and check for air entrainment in the loop. If alarms persist after flow is restored, schedule a maintenance window before the temperature reaches the interlock threshold."
  },
  {
    "doc_id": "KB-COOL-02",
    "subsystem": "cooling",
    "title": "Coolant pump seal replacement",
    "content": "A degraded pump seal shows as a slow decline in coolant flow and occasional pressure oscillation. To replace seal kit P-COOL-01: isolate the loop, relieve pressure, drain to the service level, and swap the seal following the torque sequence. Bleed air from the loop before returning to service and confirm flow stabilizes at setpoint. Log the coolant top-up volume for trend tracking."
  },
  {
    "doc_id": "KB-COOL-03",
    "subsystem": "cooling",
    "title": "Focus error caused by thermal drift",
    "content": "When focus error climbs without any optical fault, suspect thermal drift in the frame or wafer chuck. Confirm by correlating focus error against the temperature sensor: a lag of a few minutes between temperature and focus is expected. Once cooling is restored, focus error should recover within the thermal time constant. If it does not recover, escalate to optics."
  },
  {
    "doc_id": "KB-COOL-04",
    "subsystem": "cooling",
    "title": "Coolant flow interlock troubleshooting",
    "content": "A coolant flow interlock trip halts exposure to protect the system. First verify the flow sensor reading against a manual gauge to rule out a faulty sensor. If flow is genuinely low, check for a clogged filter, a failing pump, or a closed isolation valve. Do not bypass the interlock; restore flow and clear the alarm through the normal reset path."
  },
  {
    "doc_id": "KB-SRC-01",
    "subsystem": "source",
    "title": "Source power degradation",
    "content": "Sagging source power together with throughput loss points to a source module problem rather than a stage or cooling issue. Verify the power module (part P-SRC-02) output against its commanded level and inspect the collector for contamination that reduces transmitted power. A gradual decline usually means collector degradation; a sudden step suggests a module fault."
  },
  {
    "doc_id": "KB-SRC-02",
    "subsystem": "source",
    "title": "Collector contamination cleaning",
    "content": "Collector contamination reduces delivered source power and lowers wafer throughput because dose targets take longer to reach. Follow the collector inspection routine, and if reflectivity is below threshold, schedule the cleaning procedure. After cleaning, re-measure delivered power and update the dose calibration before resuming production."
  },
  {
    "doc_id": "KB-SRC-03",
    "subsystem": "source",
    "title": "Source power module fault isolation",
    "content": "To isolate a source power module fault, compare commanded versus delivered power across a range of setpoints. A consistent offset at all setpoints points to a calibration issue; instability or dropouts point to the module hardware (part P-SRC-02). Replace the module only after confirming cabling and the control signal are healthy."
  },
  {
    "doc_id": "KB-SRC-04",
    "subsystem": "source",
    "title": "Throughput loss root-cause checklist",
    "content": "Wafer throughput loss has several possible roots. Rank them: reduced source power (dose takes longer), stage settling delays, increased alarm-driven pauses, and wafer-handling slowdowns. Check delivered source power first since it is the most common cause, then review the alarm log for repeated brief stoppages that erode throughput without a single obvious fault."
  },
  {
    "doc_id": "KB-VAC-01",
    "subsystem": "vacuum",
    "title": "Vacuum pressure excursion response",
    "content": "A vacuum pressure excursion can disturb both source performance and contamination control. On a pressure rise, check for a leak at recently serviced flanges, verify pump status, and review the outgassing history if a new component was installed. Small slow rises are often outgassing; sharp rises indicate a leak or pump fault."
  },
  {
    "doc_id": "KB-VAC-02",
    "subsystem": "vacuum",
    "title": "Vacuum pump maintenance schedule",
    "content": "Vacuum pumps follow a preventive schedule based on run hours and observed base pressure. Track base pressure over time; a rising trend at constant load signals approaching service need. Perform the scheduled service before base pressure crosses the action limit to avoid an unplanned interruption."
  },
  {
    "doc_id": "KB-VAC-03",
    "subsystem": "vacuum",
    "title": "Leak detection procedure",
    "content": "For suspected vacuum leaks, isolate sections and observe the pressure rate of rise. Use the tracer-gas method around suspect flanges. Document which section shows the fastest rise. Re-torque or reseal the identified flange, then confirm base pressure returns to nominal before releasing the tool."
  },
  {
    "doc_id": "KB-WFR-01",
    "subsystem": "wafer handler",
    "title": "Wafer handler alarm recovery",
    "content": "A single transient wafer-handler alarm that auto-recovers is usually benign, often a sensor debounce or a marginal grip event. Review the handler log for repetition. Isolated events need no action beyond logging; repeated events in the same position indicate a mechanical or sensor problem needing inspection."
  },
  {
    "doc_id": "KB-WFR-02",
    "subsystem": "wafer handler",
    "title": "Wafer chuck contamination",
    "content": "Chuck contamination causes clamping errors and can manifest as focus or overlay noise. Inspect the chuck surface, run the cleaning routine, and verify flatness after cleaning. Persistent clamping errors after cleaning suggest a worn chuck or a vacuum-clamp leak."
  },
  {
    "doc_id": "KB-WFR-03",
    "subsystem": "wafer handler",
    "title": "Robot handoff timing errors",
    "content": "Handoff timing errors between the wafer robot and the chuck slow throughput and can trigger alarms. Check the handoff position calibration and the grip confirmation sensor. Small timing drifts are usually recalibrated in software; repeated grip failures point to worn end-effector pads."
  },
  {
    "doc_id": "KB-ALM-01",
    "subsystem": "general",
    "title": "Alarm flood triage",
    "content": "During an alarm flood, group alarms by subsystem and timestamp rather than reacting to each individually. The earliest alarm in a cluster is usually the root cause and later alarms are consequences. Silence non-safety nuisance alarms only after the root cause is identified, never before."
  },
  {
    "doc_id": "KB-ALM-02",
    "subsystem": "general",
    "title": "Alarm count trend interpretation",
    "content": "A rising alarm-count trend, even below the alert threshold, is an early warning that a subsystem is degrading. Correlate the alarm-count rise with sensor trends: alarms rising alongside temperature suggest cooling; alongside overlay suggest the stage. Use the trend to schedule proactive inspection."
  },
  {
    "doc_id": "KB-PM-01",
    "subsystem": "general",
    "title": "Preventive maintenance planning",
    "content": "Preventive maintenance scheduling balances time-since-maintenance against observed health indicators. A tool well past its nominal interval with a declining health score should be prioritized. Confirm required parts are in stock and a qualified specialist is available before opening a maintenance window to avoid extended downtime."
  },
  {
    "doc_id": "KB-PM-02",
    "subsystem": "general",
    "title": "Time-since-maintenance risk factors",
    "content": "As time since maintenance grows, the probability of drift-related faults increases, particularly on the reticle stage and cooling loop. Treat a high time-since-maintenance value combined with any anomalous sensor trend as an elevated-risk condition warranting earlier intervention."
  },
  {
    "doc_id": "KB-HND-01",
    "subsystem": "general",
    "title": "Shift handover best practices",
    "content": "An effective shift handover separates verified facts from suggested actions. State each open issue, the supporting evidence, the owner, and the current status. Avoid mixing speculation with confirmed observations so the incoming shift can act on facts and evaluate suggestions independently."
  },
  {
    "doc_id": "KB-HND-02",
    "subsystem": "general",
    "title": "Escalation criteria",
    "content": "Escalate when a required specialist is unavailable, a needed part is out of stock with a long lead time, or a health score continues to decline after corrective action. Escalation should include the evidence gathered so the next tier does not repeat the investigation from scratch."
  },
  {
    "doc_id": "KB-FOC-01",
    "subsystem": "cooling",
    "title": "Focus error versus overlay error differentiation",
    "content": "Focus error and overlay error have different root causes and should not be confused. Focus error most often tracks thermal and optical issues, while overlay error tracks stage and alignment issues. When both rise together, thermal drift affecting both the focal plane and stage positioning is a common shared cause worth checking first."
  },
  {
    "doc_id": "KB-THR-01",
    "subsystem": "source",
    "title": "Wafer throughput baseline and deviation",
    "content": "Establish a wafer-throughput baseline per tool and product. A sustained deviation below baseline, once wafer mix is accounted for, indicates a developing problem. Pair throughput deviation with source power and alarm trends to distinguish a source issue from a handling or stage issue."
  },
  {
    "doc_id": "KB-VIB-01",
    "subsystem": "reticle stage",
    "title": "Vibration threshold and early warning",
    "content": "Vibration rising steadily toward its threshold is a reliable early-warning indicator for stage mechanical wear. Trend the vibration RMS; a monotonic rise over multiple shifts warrants inspection before the threshold trips, creating time to plan rather than react."
  },
  {
    "doc_id": "KB-DRIFT-01",
    "subsystem": "general",
    "title": "Slow sensor drift detection",
    "content": "Slow sensor drift is dangerous precisely because it stays below alarm limits until it suddenly does not. Anomaly detection on combinations of sensors catches drift earlier than single-sensor thresholds. When an anomaly model flags drift, review the contributing sensors to localize the subsystem."
  },
  {
    "doc_id": "KB-INC-01",
    "subsystem": "general",
    "title": "Incident prioritization framework",
    "content": "Prioritize incidents by severity and by remaining useful life. A high-severity incident on a machine with low remaining useful life demands immediate attention; a low-severity transient that auto-recovered can be logged and monitored. Always attach the supporting sensor evidence to the incident record so prioritization is traceable."
  },
  {
    "doc_id": "KB-RUL-01",
    "subsystem": "general",
    "title": "Remaining useful life interpretation",
    "content": "A short predicted remaining useful life means intervention should be planned now, while a long value supports normal operation. Treat remaining useful life as a planning aid alongside the health score and failure risk, not as a guarantee. Confirm the prediction against the underlying sensor trends before acting on it."
  }
]

print(f'Loaded {len(KNOWLEDGE)} synthetic maintenance documents.')
subs = {}
for r in KNOWLEDGE:
    subs[r['subsystem']] = subs.get(r['subsystem'], 0) + 1
print('By subsystem:', subs)

## 3. Embed the documents and build the FAISS index

Each document becomes a vector (a list of numbers capturing its meaning). The first run downloads the ~90 MB embedding model.

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [f"{r['title']}. {r['content']}" for r in KNOWLEDGE]
emb = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True).astype('float32')

index = faiss.IndexFlatIP(emb.shape[1])   # cosine similarity on normalized vectors
index.add(emb)
print(f'Indexed {index.ntotal} documents, each as a {emb.shape[1]}-dim vector.')

## 4. Semantic search function

Embed the query the same way, then ask FAISS for the nearest documents.

In [ ]:
def search(query, k=3):
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    scores, idx = index.search(q, k)
    results = []
    for score, i in zip(scores[0], idx[0]):
        r = dict(KNOWLEDGE[i]); r['score'] = round(float(score), 3)
        results.append(r)
    return results

def show(query, k=3):
    print(f'QUERY: {query}\n')
    for r in search(query, k):
        print(f"  [{r['score']}] {r['doc_id']} ({r['subsystem']}) — {r['title']}")
    print()

print('search() ready.')

## 5. Try it — semantic retrieval in action

Notice these queries use *everyday* words, not the exact terms in the documents. Semantic search still finds the right ones.

In [ ]:
show('the machine is getting hot and the image is blurry')
show('wafers are coming out slower than usual')
show('how do I hand over open issues to the next shift')
show('the alignment is slowly getting worse over the day')

## 6. Connect it to a real sensor situation

This mirrors what the **Knowledge agent** does in the main project: it takes the symptoms detected on a machine and retrieves the matching procedure automatically.

In [ ]:
# Example: EUV-03 in the project shows rising temperature + focus error (a cooling fault)
machine_symptoms = 'temperature rising and focus error increasing with more alarms'
print('Detected symptoms on LITHO-EUV-03:')
print(' ', machine_symptoms, '\n')
print('Retrieved guidance the agent would cite:')
for r in search(machine_symptoms, k=2):
    print(f"\n[{r['score']}] {r['doc_id']} — {r['title']}")
    print('  ', r['content'])

## 7. (Optional) Save the index to Google Drive

So you can reuse it later without rebuilding. Uncomment and run.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import json
# faiss.write_index(index, '/content/drive/MyDrive/lithoops_rag_index.faiss')
# json.dump(KNOWLEDGE, open('/content/drive/MyDrive/lithoops_rag_meta.json','w'), indent=2)
# print('Saved index + metadata to your Google Drive.')

## What you just built

- A **real semantic search engine** over a maintenance knowledge base, using free
  local embeddings and a FAISS vector store — no API keys, no cost.
- It retrieves by **meaning**, so operator-style plain-language symptoms find the
  right technical procedure.
- This is the retrieval half of a **RAG agent**. In **Stage B** we add a small
  free LLM that reads these retrieved documents plus the sensor evidence and
  writes a grounded, cited shift-handover report.

**For your portfolio / interview:** you can now say your Knowledge agent uses
sentence-transformer embeddings with FAISS for semantic retrieval — and explain
*why* semantic beats keyword search for scattered technical knowledge.